# LLM PKM Harness — Proof of Concept

This notebook demonstrates the **deterministic core** of the compiled-wiki harness pattern ([Karpathy LLM Wiki](https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f)) without calling an LLM API.

**What this POC covers:**
1. Vault structure (`raw/`, `wiki/`, `index.md`, `log.md`)
2. **Ingest** — source → wiki pages + index + log
3. **Query** — index-first navigation → relevant pages
4. **Lint** — orphans, dead links, missing index entries

**What you plug in later:** an LLM agent replaces the structured `ingest_source()` arguments with real extraction from raw markdown.

See also: [Obsidian setup guide](../obsidian_setup_guide.md) | [Main research](../README.md)

## 0. Setup

In [ ]:
import shutil
import sys
from pathlib import Path

POC_DIR = Path.cwd()
if POC_DIR.name != "poc":
    POC_DIR = Path("researches/llm_harnessing_for_pkm/poc").resolve()

sys.path.insert(0, str(POC_DIR))
from harness import WikiVault, LintIssue

VAULT_ROOT = POC_DIR / "sample_vault"
if VAULT_ROOT.exists():
    shutil.rmtree(VAULT_ROOT)

vault = WikiVault(VAULT_ROOT)
vault.ensure_structure()
print(f"Vault created at: {VAULT_ROOT}")

## 1. Vault structure

The harness expects three layers:
- `raw/` — immutable sources (human adds, agent reads)
- `wiki/` — compiled pages (agent writes)
- `index.md` + `log.md` — navigation and audit trail

In [ ]:
for path in sorted(VAULT_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(VAULT_ROOT))

## 2. Add a raw source

In Obsidian, you'd clip an article to `raw/`. Here we write a sample markdown source.

In [ ]:
RAW_ARTICLE = """# Retrieval-Augmented Generation

RAG retrieves document chunks at query time and injects them into an LLM prompt.

Karpathy's compiled wiki pattern instead synthesizes at ingest time, building
a persistent markdown wiki with cross-references.

Tools like qmd add hybrid search when the wiki outgrows index.md navigation.
"""

raw_path = vault.raw_dir / "rag-vs-compiled-wiki.md"
raw_path.write_text(RAW_ARTICLE, encoding="utf-8")
print(f"Wrote: {raw_path.relative_to(VAULT_ROOT)}")

## 3. Ingest pipeline

In production, an **LLM agent** reads the raw file and:
1. Extracts concepts, entities, claims
2. Creates/updates wiki pages
3. Updates `index.md`
4. Appends to `log.md`

This POC simulates that extraction with structured arguments.

In [ ]:
created = vault.ingest_source(
    raw_filename="rag-vs-compiled-wiki.md",
    title="RAG vs Compiled Wiki",
    summary="Comparison of query-time RAG and ingest-time compiled wiki patterns.",
    concepts=[
        ("Retrieval-Augmented Generation", "Query-time retrieval over chunked documents."),
        ("Compiled Wiki", "Ingest-time synthesis into interlinked markdown pages."),
        ("qmd", "Local hybrid search for markdown when index.md is too large."),
    ],
    entities=[
        ("Andrej Karpathy", "Proposed the LLM Wiki compiled knowledge pattern."),
    ],
)

print(f"Created/updated {len(created)} pages:")
for p in created:
    print(f"  - {p.relative_to(VAULT_ROOT)}")

In [ ]:
print("=== index.md ===")
print(vault.index_path.read_text())

In [ ]:
print("=== log.md ===")
print(vault.log_path.read_text())

## 4. Query pipeline (index-first)

The agent reads `index.md` first, picks relevant pages, then reads them.
No embeddings required at small scale.

In [ ]:
query = "compiled wiki synthesis ingest"
matches = vault.query(query, top_k=5)

print(f"Query: {query!r}\n")
for entry, score in matches:
    print(f"  [{score:.2f}] [[{entry.title}]] ({entry.category}) — {entry.summary}")

In [ ]:
pages = vault.read_pages_for_query("RAG retrieval query time", top_k=2)

for entry, content in pages:
    print(f"\n{'='*60}")
    print(f"Page: [[{entry.title}]]")
    print(f"{'='*60}")
    print(content[:500] + ("..." if len(content) > 500 else ""))

## 5. Lint pipeline

Periodic health checks: orphans, dead links, pages missing from index.

In [ ]:
# Introduce a deliberate issue: page with dead link, not in index
bad_page = vault.wiki_dir / "derived" / "orphan-note.md"
bad_page.write_text(
    "# Orphan Note\n\nLinks to [[Nonexistent Page]] and [[RAG vs Compiled Wiki]].\n",
    encoding="utf-8",
)

issues = vault.lint()
print(f"Found {len(issues)} issue(s):\n")
for issue in issues:
    loc = Path(issue.path).relative_to(VAULT_ROOT) if issue.path else "?"
    print(f"  [{issue.kind}] {issue.message} ({loc})")

In [ ]:
vault.append_log("lint", f"{len(issues)} issues found")
print(vault.log_path.read_text())

## 6. Where the LLM plugs in

| Harness step | Deterministic (this POC) | LLM agent (production) |
| --- | --- | --- |
| **Ingest** | `ingest_source(...)` with hand-supplied concepts | Read `raw/*.md`, extract entities/concepts, write pages |
| **Query** | Keyword match on index | Read index, reason about relevance, synthesize cited answer |
| **Lint** | Regex wikilink + index checks | Same checks + semantic contradiction detection, gap suggestions |
| **Schema** | Python module | `CLAUDE.md` / `AGENTS.md` at vault root |

The POC shows that **~40% of the harness is deterministic** — folder I/O, index maintenance, logging, link validation. The LLM adds synthesis, judgment, and cross-document integration.

In [ ]:
def llm_ingest_placeholder(vault: WikiVault, raw_filename: str) -> None:
    """
    Production replacement sketch:

    1. raw_text = (vault.raw_dir / raw_filename).read_text()
    2. extraction = llm.extract(raw_text, schema=AGENTS_MD)
    3. vault.ingest_source(raw_filename, **extraction)
    """
    raise NotImplementedError("Connect your LLM agent here (Claude Code, Cursor, etc.)")

print(llm_ingest_placeholder.__doc__)

## 7. Full vault tree (after POC run)

In [ ]:
for path in sorted(VAULT_ROOT.rglob("*")):
    if path.is_file():
        size = path.stat().st_size
        print(f"{path.relative_to(VAULT_ROOT)} ({size} bytes)")